In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, Input, MultiHeadAttention, LayerNormalization, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import itertools
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

print("GPU is available" if tf.config.list_physical_devices('GPU') else "NOT AVAILABLE")

In [ ]:
# --- Configuration ---
class Config:
    MIX_VOCAB_SIZE = 16000  # Reduced from 32000
    EMBED_DIM = 192  # Keep
    FF_DIM = 512  # Could reduce from 1024 to speed up
    NUM_HEADS = 8  # Keep
    
    NUM_ENCODER_LAYERS = 5  # Keep
    NUM_DECODER_LAYERS = 5  # Keep
    DROPOUT_RATE = 0.15  # Slightly increase for regularization
    MAX_LENGTH = 96  # Keep
    BATCH_SIZE = 128  # Increase if GPU memory allows
    BUFFER_SIZE = 20000  # Keep
    EPOCHS = 75  # Keep, but use early stopping (you have it)
    WARMUP_STEPS = 2000  # Reduce proportionally to dataset size
    
    # Use absolute paths or relative to execution
    DATA_DIR = "./"
    SP_DIR = os.path.join(DATA_DIR, "sp_data")
    BPE_MODEL_PREFIX = os.path.join(DATA_DIR, "bpe_multi")
    CSV_PATH = os.path.join(DATA_DIR, "data.csv")
    TF_RECORD_TRAIN = os.path.join(DATA_DIR, "train.tfrecord")
    TF_RECORD_VAL = os.path.join(DATA_DIR, "val.tfrecord")

    IS_TRAINING = True

config = Config()

In [ ]:
def visualize_data(df):
    print("--- Visualizing Data ---")
    # Histogram of sentence length in tokens
    # Note: Just splitting by space as a rough approximation similar to original notebook
    eng_length = [len(str(sentence).split()) for sentence in df["english"].dropna()]
    esp_length = [len(str(sentence).split()) for sentence in df["spanish"].dropna()]

    plt.figure(figsize=(10, 6))
    plt.hist(eng_length, label="eng", color="red", alpha=0.33, bins=50)
    plt.hist(esp_length, label="esp", color="blue", alpha=0.33, bins=50)
    
    plt.yscale("log") 
    plt.legend()
    plt.title("Examples count vs Token length")
    plt.xlabel("Token Length")
    plt.ylabel("Count (Log Scale)")
    
    output_path = os.path.join(config.DATA_DIR, "dataset_visualization.png")
    plt.savefig(output_path)
    print(f"Visualization saved to {output_path}")
    plt.close()

| Sample | Desired Lang | Source Input (Encoder) | Target Input (Decoder) | Target Label (Loss) |
| :--- | :--- | :--- | :--- | :--- |
| **1 (En$\to$Es)** | Spanish | `[<2es>, The, car, <EOS>, <PAD>]` | `[<BOS>, El, coche, <PAD>]` | `[El, coche, <EOS>, <PAD>]` |
| **2 (En$\to$Fr)** | French | `[<2fr>, The, car, <EOS>, <PAD>]` | `[<BOS>, La, voiture, <PAD>]`| `[La, voiture, <EOS>, <PAD>]`|
| **3 (En$\to$De)** | German | `[<2de>, The, car, <EOS>, <PAD>]` | `[<BOS>, Das, Auto, <PAD>]` | `[Das, Auto, <EOS>, <PAD>]` |

In [ ]:
# --- 1. Data Preparation & BPE Training ---
def clean_dataset(df):
    print("Initial shape of dataset: ", df.shape)
    df = df.dropna()
    df = df.reset_index(drop=True)
    print("Dataset cleaned. Shape: ", df.shape)
    return df


def train_bpe(df):
    try:
        combined_iterator = itertools.chain(
            df['english'].astype(str),
            df['spanish'].astype(str)
        )
        if not os.path.exists(config.BPE_MODEL_PREFIX + ".json"):
            print("Training BPE model...")
            bpe_tokeniser = Tokenizer(models.BPE(unk_token="<unk>"))
            normalizer = normalizers.Sequence([normalizers.NFD(), normalizers.StripAccents()])
            pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Whitespace(), pre_tokenizers.Digits(individual_digits=True)])
            bpe_trainer = trainers.BpeTrainer(
                vocab_size=config.MIX_VOCAB_SIZE,
                special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
                show_progress=True,
            )
            bpe_tokeniser.normalizer = normalizer
            bpe_tokeniser.pre_tokenizer = pre_tokenizer
            bpe_tokeniser.train_from_iterator(combined_iterator, trainer=bpe_trainer)
            bpe_tokeniser.save(config.BPE_MODEL_PREFIX + ".json")

            print(f"BPE model saved to {config.BPE_MODEL_PREFIX}.json")
            return bpe_tokeniser
        else:
            bpe_tokeniser = Tokenizer.from_file(config.BPE_MODEL_PREFIX + ".json")

            print(f"BPE model found at {config.BPE_MODEL_PREFIX}.json")
            return bpe_tokeniser
    except Exception as e:
        print(f"Error training BPE model: {e}")


def tokenize_dataset(df, bpe_tokeniser):
    # 1. Prepare Data (Vectorized String Operations)
    # Using specific columns avoids copying the whole DF
    print("Preparing string data...")
    
    # Process French
    df_es = df[["english", "spanish"]].dropna().copy()
    df_es["source"] = df_es["english"].str.lower()
    df_es["target"] = df_es["spanish"].str.lower()

    # Concatenate once
    final_df = pd.concat([df_es[["source", "target"]]], ignore_index=True)
    
    print(f"Final shape of raw dataset: {final_df.shape}")

    # 2. Batch Tokenization (The Speedup)
    # Convert to list for the tokenizer (much faster than iterating series)
    source_texts = final_df["source"].tolist()
    # Add BOS/EOS to targets via list comprehension (faster than pandas apply)
    target_texts = ["<bos> " + t + " <eos>" for t in final_df["target"].tolist()]

    print("Tokenizing in batches...")
    
    # encode_batch runs in Rust and uses multi-threading
    source_encodings = bpe_tokeniser.encode_batch(source_texts)
    target_encodings = bpe_tokeniser.encode_batch(target_texts)

    # 3. Extract IDs and Filter (Vectorized)
    # Extract IDs from the Encoding objects
    final_df["source_ids"] = [e.ids for e in source_encodings]
    final_df["target_ids"] = [e.ids for e in target_encodings]
    
    # Calculate lengths efficiently
    final_df["src_len"] = final_df["source_ids"].map(len)
    final_df["tgt_len"] = final_df["target_ids"].map(len)

    # Filter using boolean masking (Vectorized)
    mask = (final_df["src_len"] <= config.MAX_LENGTH) & \
           (final_df["tgt_len"] <= config.MAX_LENGTH + 1)
    
    filtered_df = final_df.loc[mask, ["source_ids", "target_ids"]].rename(
        columns={"source_ids": "source", "target_ids": "target"}
    ).reset_index(drop=True)

    print(f"Filtered and Tokenized {len(filtered_df)} samples")
    return filtered_df


In [ ]:
# --- 2. TFRecord Pipeline ---
def serialize_example(src, trg):
    feature = {
        "source": tf.train.Feature(int64_list=tf.train.Int64List(value=src)),
        "target": tf.train.Feature(int64_list=tf.train.Int64List(value=trg)),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()

def parse_fn(example):
    feature_description = {
        "source": tf.io.VarLenFeature(tf.int64),
        "target": tf.io.VarLenFeature(tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)
    src = tf.sparse.to_dense(example["source"])
    trg = tf.sparse.to_dense(example["target"])
    return src, trg

def create_tfrecords(dataset, filename):
    print(f"Writing {filename}...")
    with tf.io.TFRecordWriter(filename) as writer:
        for s, t in dataset.itertuples(index=False):
            writer.write(serialize_example(s, t))

def split_target(src, trg):
    return src, trg[:-1], trg[1:]

In [ ]:
# --- 3. Model Architecture ---
class PositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.embed_dim = embed_dim
        self.pos_encoding = self.positional_encoding(sequence_length, embed_dim)

    def get_angles(self, pos, i, d_model):
        angles = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        return pos * angles

    def positional_encoding(self, position, d_model):
        angle_rads = self.get_angles(
            np.arange(position)[:, np.newaxis],
            np.arange(d_model)[np.newaxis, :],
            d_model
        )
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def call(self, inputs):
        length = tf.shape(inputs)[1]
        return inputs + self.pos_encoding[:, :length, :]

class TransformerEncoder(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, drop_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(drop_rate)
        self.dropout2 = Dropout(drop_rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerDecoder(Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, drop_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att1 = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.att2 = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(drop_rate)
        self.dropout2 = Dropout(drop_rate)
        self.dropout3 = Dropout(drop_rate)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)],
            axis=0
        )
        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs, training=False):
        causal_mask = self.get_causal_attention_mask(inputs)
        att1_output = self.att1(inputs, inputs, attention_mask=causal_mask)
        att1_output = self.dropout1(att1_output, training=training)
        out1 = self.layernorm1(inputs + att1_output)
        
        att2_output = self.att2(out1, encoder_outputs)
        att2_output = self.dropout2(att2_output, training=training)
        out2 = self.layernorm2(out1 + att2_output)
        
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        return self.layernorm3(out2 + ffn_output)

def build_transformer(config):
    # Encoder
    encoder_inputs = Input(shape=(None,), dtype="int64", name="encoder_inputs")
    x = Embedding(config.MIX_VOCAB_SIZE, config.EMBED_DIM)(encoder_inputs)
    x = PositionalEmbedding(config.MAX_LENGTH, config.EMBED_DIM)(x)
    x = Dropout(config.DROPOUT_RATE)(x)
    
    for _ in range(config.NUM_ENCODER_LAYERS):
        x = TransformerEncoder(config.EMBED_DIM, config.NUM_HEADS, config.FF_DIM, config.DROPOUT_RATE)(x)
    encoder_outputs = x
    
    # Decoder
    decoder_inputs = Input(shape=(None,), dtype="int64", name="decoder_inputs")
    x = Embedding(config.MIX_VOCAB_SIZE, config.EMBED_DIM)(decoder_inputs)
    x = PositionalEmbedding(config.MAX_LENGTH, config.EMBED_DIM)(x)
    x = Dropout(config.DROPOUT_RATE)(x)
    
    for _ in range(config.NUM_DECODER_LAYERS):
        x = TransformerDecoder(config.EMBED_DIM, config.FF_DIM, config.NUM_HEADS, config.DROPOUT_RATE)(x, encoder_outputs)
        
    decoder_outputs = Dense(config.MIX_VOCAB_SIZE, activation="softmax", dtype='float32')(x)
    
    return Model([encoder_inputs, decoder_inputs], decoder_outputs, name="transformer")

class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

In [ ]:
dataset = pd.read_csv("./data.csv")
dataset = clean_dataset(dataset)
dataset = dataset[:]

In [ ]:
bpe_tok = train_bpe(dataset)
bpe_tok.encode("Hello World").tokens

In [ ]:
train_df, valid_df = train_test_split(dataset, test_size=0.2, random_state=42)
train_df.shape, valid_df.shape

In [ ]:
train_df = tokenize_dataset(train_df, bpe_tok)
valid_df = tokenize_dataset(valid_df, bpe_tok)

In [ ]:
create_tfrecords(train_df, "./train.tfrecord")
create_tfrecords(valid_df, "./val.tfrecord")

In [ ]:
train_dataset_raw = (
    tf.data.TFRecordDataset("./train.tfrecord")
    .map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .map(split_target, num_parallel_calls=tf.data.AUTOTUNE)
)

valid_dataset_raw = (
    tf.data.TFRecordDataset("./val.tfrecord")
    .map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .map(split_target, num_parallel_calls=tf.data.AUTOTUNE)
)

In [ ]:
BUFFER_SIZE = 10000

def element_length_func(inputs, targets):
    return tf.shape(inputs["encoder_inputs"])[0]

def format_dataset(src, trg_in, trg_out):
    return ({"encoder_inputs": src, "decoder_inputs": trg_in}, trg_out)


bucket_boundaries = [25, 50, 75, 100]
bucket_batch_sizes = [config.BATCH_SIZE] * (len(bucket_boundaries) + 1)

train_ds = (
    train_dataset_raw
    .map(format_dataset, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(BUFFER_SIZE)
    .bucket_by_sequence_length(
        element_length_func=element_length_func,
        bucket_boundaries=bucket_boundaries,
        bucket_batch_sizes=bucket_batch_sizes,
        padding_values=(
            {"encoder_inputs": tf.constant(0, dtype=tf.int64),
             "decoder_inputs": tf.constant(0, dtype=tf.int64)},
            tf.constant(0, dtype=tf.int64)
        ),
        drop_remainder=True,
    )
    .prefetch(tf.data.AUTOTUNE)
)

valid_ds = (
    valid_dataset_raw
    .map(format_dataset, num_parallel_calls=tf.data.AUTOTUNE)
    .bucket_by_sequence_length(
        element_length_func=element_length_func,
        bucket_boundaries=bucket_boundaries,
        bucket_batch_sizes=bucket_batch_sizes,
        padding_values=(
            {"encoder_inputs": tf.constant(0, dtype=tf.int64),
             "decoder_inputs": tf.constant(0, dtype=tf.int64)},
            tf.constant(0, dtype=tf.int64)
        ),
        drop_remainder=True,
    )
    .prefetch(tf.data.AUTOTUNE)
)

print("Data pipelines created successfully using the stable API.")

In [ ]:
import tensorflow as tf

# Initialize the standard loss function with reduction='none'
# We need 'none' so we can apply the mask manually before summing
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False, 
    reduction='none'
)

def masked_loss(real, pred):
    # 1. Create a mask: True for valid tokens, False for 0 (padding)
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    
    # 2. Calculate standard loss
    # real shape: (batch_size, seq_len)
    # pred shape: (batch_size, seq_len, vocab_size)
    loss_ = loss_object(real, pred)

    # 3. Apply the mask
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    # 4. Normalize by the number of VALID tokens (not just batch size)
    # This prevents the loss from being artificially small due to padding
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)


def masked_accuracy(real, pred):
    # 1. Get the index of the highest probability prediction
    pred_ids = tf.argmax(pred, axis=2)
    
    # 2. Check equality (True/False)
    # Cast real to match pred_ids dtype (int64) if necessary
    real = tf.cast(real, pred_ids.dtype)
    is_correct = tf.equal(real, pred_ids)

    # 3. Create mask for padding (0)
    mask = tf.math.logical_not(tf.math.equal(real, 0))

    # 4. Filter out padding from the accuracy check
    # logical_and ensures we only count correct predictions if they are NOT padding
    is_correct = tf.math.logical_and(mask, is_correct)

    # 5. Calculate average
    is_correct = tf.cast(is_correct, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    
    return tf.reduce_sum(is_correct) / tf.reduce_sum(mask)

In [ ]:
transformer = build_transformer(config)
learning_rate = CustomSchedule(config.EMBED_DIM)
optimizer = Adam(
    learning_rate, 
    beta_1=0.9, 
    beta_2=0.98, 
    epsilon=1e-9
)

transformer.compile(
    optimizer=optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy]
)

transformer.summary()

In [ ]:
model_name = "transformer_model_binary.weights.h5"

if config.IS_TRAINING:
    # Callback to save the best model during training
    checkpoints = ModelCheckpoint(
        model_name,  # Corrected filename extension
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True, 
    )
    
    # Callback for early stopping
    early_stop = EarlyStopping(
        patience=15,
        monitor="val_loss",
        mode="min",
        restore_best_weights=True,
    )

    # Training the model
    print("--- Starting Model Training ---")
    history = transformer.fit(
        train_ds,
        epochs=config.EPOCHS,
        validation_data=valid_ds,
        callbacks=[checkpoints, early_stop],
    )

    print("\\n--- Model Training Finished ---")
    transformer.save("transformer_model_binary.keras")

    print("Loading best weights from training...")
    transformer.load_weights(model_name)

else:
    # If not training, attempt to load a pre-existing model
    try:
        print("Loading pre-trained weights...")
        transformer.load_weights(model_name)
    except (FileNotFoundError, tf.errors.NotFoundError):
        print(f"Error: Model weights file {model_name} not found.")
        print("Please set config.is_training to True to train the model first.")